In [ ]:
import jsonimport boto3import osimport timeimport urllib3import uuidfrom datetime import datetime, timezonefrom urllib.parse import urlparsefrom boto3.dynamodb.conditions import Key# clientss3 = boto3.client("s3")ddb = boto3.resource("dynamodb")athena = boto3.client("athena")http = urllib3.PoolManager()# envBUCKET = os.environ["ATHENA_RESULT_BUCKET"]TABLE_NAME = os.environ["USER_TABLE"]DB = os.environ["RAW_DB"]RAW_TABLE = os.environ["RAW_TABLE"]GSI = os.environ["USER_TOKEN_GSI"]API = os.environ["API_ENDPOINT"]table = ddb.Table(TABLE_NAME)# -----------------------------------------# Athena 실행# -----------------------------------------def run_athena(sql):    print("ATHENA QUERY =", sql)    res = athena.start_query_execution(        QueryString=sql,        QueryExecutionContext={"Database": DB},        ResultConfiguration={"OutputLocation": f"s3://{BUCKET}/"}    )    qid = res["QueryExecutionId"]    for _ in range(60):        status = athena.get_query_execution(QueryExecutionId=qid)        state = status["QueryExecution"]["Status"]["State"]        if state in ["SUCCEEDED", "FAILED", "CANCELLED"]:            break        time.sleep(1)    if state != "SUCCEEDED":        raise Exception(f"Athena failed: {state}")    return athena.get_query_results(QueryExecutionId=qid)# -----------------------------------------# GPS 문자열 → 객체 변환# -----------------------------------------def gps_to_obj(gps):    if not gps:        return None    try:        lat, lon = gps.split(",")        return {"lat": float(lat), "lon": float(lon)}    except:        return None# -----------------------------------------# 최신 GPS 조회# -----------------------------------------def get_latest_gps(tokens):    if not tokens:        return {}    ids = ",".join([f"'{u}'" for u in tokens])    sql = f"""    SELECT user_token,           max_by(coarse_gps, event_ts)    FROM {RAW_TABLE}    WHERE coarse_gps IS NOT NULL      AND user_token IN ({ids})      AND dt >= current_date - interval '1' day    GROUP BY user_token    """    result = run_athena(sql)    gps_map = {}    rows = result["ResultSet"]["Rows"][1:]    for r in rows:        user = r["Data"][0]["VarCharValue"]        gps = r["Data"][1]["VarCharValue"]        gps_map[user] = gps    print("GPS MAP =", gps_map)    return gps_map# -----------------------------------------# friends 파싱# -----------------------------------------def parse_friends(raw):    if not raw:        return []    if isinstance(raw, list):        return raw    if isinstance(raw, dict) and "L" in raw:        return [x["S"] for x in raw["L"] if "S" in x]    return []# -----------------------------------------# user_token → user_id + friends# -----------------------------------------def get_user_and_friends(tokens):    user_map = {}    friend_map = {}    for token in tokens:        res = table.query(            IndexName=GSI,            KeyConditionExpression=Key("user_token").eq(token)        )        items = res.get("Items", [])        if not items:            continue        item = items[0]        user_id = item["user_id"]        if "friends" not in item:            full = table.get_item(                Key={                    "user_id": user_id,                    "token_type": "USER"                }            )            item = full.get("Item", {})        user_map[token] = user_id        friend_map[user_id] = parse_friends(item.get("friends"))    print("USER MAP =", user_map)    print("FRIEND MAP =", friend_map)    return user_map, friend_map# -----------------------------------------# user_id → token# -----------------------------------------def get_tokens_from_user_ids(user_ids):    token_map = {}    if not user_ids:        return token_map    keys = [        {"user_id": uid, "token_type": "USER"}        for uid in user_ids    ]    res = ddb.batch_get_item(        RequestItems={            TABLE_NAME: {                "Keys": keys            }        }    )    items = res["Responses"].get(TABLE_NAME, [])    for item in items:        token_map[item["user_id"]] = item.get("user_token")    print("FRIEND TOKEN MAP =", token_map)    return token_map# -----------------------------------------# S3 key 추출# -----------------------------------------def extract_key(url):    if url.startswith("s3://"):        return url.replace("s3://","").split("/",1)[1]    parsed = urlparse(url)    path = parsed.path.lstrip("/")    return path.split("/",1)[1]# -----------------------------------------# Lambda Handler# -----------------------------------------def lambda_handler(event, context):    print("EVENT =", json.dumps(event))    url = event["detail"]["outputLocation"]    key = extract_key(url)    print("S3 KEY =", key)    if not key.endswith(".csv"):        return {"status": "skip_non_csv"}    obj = s3.get_object(Bucket=BUCKET, Key=key)    content = obj["Body"].read().decode()    tokens = [u.strip() for u in content.split("\n") if u.strip()]    print("TARGET TOKENS =", tokens)    if not tokens:        return {"status": "no_target_users"}    user_map, friend_map = get_user_and_friends(tokens)    if not user_map:        return {"status": "no_valid_users"}    friend_ids = set()    for flist in friend_map.values():        friend_ids.update(flist)    friend_token_map = get_tokens_from_user_ids(friend_ids)    all_tokens = set(tokens)    all_tokens.update(friend_token_map.values())    gps_map = get_latest_gps(list(all_tokens))    payloads = []    for token in tokens:        if token not in user_map:            continue        user_id = user_map[token]        friends_payload = []        for fid in friend_map.get(user_id, []):            ftoken = friend_token_map.get(fid)            friends_payload.append({                "friend_user_id": fid,                "friend_gps": gps_to_obj(gps_map.get(ftoken))            })        payloads.append({            "event_id": str(uuid.uuid4()),            "critical_user_id": user_id,            "critical_gps": gps_to_obj(gps_map.get(token)),            "friends": friends_payload,            "occurred_at": datetime.now(timezone.utc).isoformat()        })    for p in payloads:        print("POST PAYLOAD =", json.dumps(p, indent=2))        res = http.request(            "POST",            API,            body=json.dumps(p).encode(),            headers={"Content-Type": "application/json"}        )        print("API STATUS =", res.status)        print("API RESPONSE =", res.data.decode())    return {        "status": "success",        "targets": len(payloads)    }